# Notebook 12 - Experiment 20: Dual Framework Bias Audit
### Novelty 6
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

In Experiment 15 I computed the fairness gaps by hand. For multi-class Average Odds
Difference that turned out to be fiddly and easy to get wrong, so here I recompute
everything with two toolkits instead: IBM's AIF360 and Microsoft's Fairlearn.

The point of using both is a cross-check. If two independently written libraries
land on the same number, I'm confident it's real. If they disagree, I report the gap
rather than quietly going with whichever looks nicer.

AIF360 also gives me the Theil Index and a Consistency score, which pick up inequality
*inside* a subgroup that a group average hides — a model can look fine on sarcasm
overall while getting one cluster of sarcasm posts almost always wrong.

No training here. Fills **Table 9**.

> **Install note:** AIF360 sometimes needs a runtime restart after installing. If
> Cell 1 errors the first time, run it, restart the runtime, then run it again — this
> caught me out a few times.

## Cell 1: Setup and Install Frameworks

In [ ]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json

!pip install -q aif360 fairlearn

try:
    from aif360.datasets import BinaryLabelDataset
    print("AIF360 available")
except ImportError:
    print("AIF360 failed. Restart the runtime and run this cell again.")

try:
    from fairlearn.metrics import MetricFrame
    print("Fairlearn available")
except ImportError:
    print("Fairlearn failed to import.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 4.4 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded. [V2 FIXED: Fairlearn EOD, Theil index]
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 5.2 MB/s eta 0:00:00
AIF360 available
Fairlearn available


## Cell 2: Load Predictions

The audit runs on the final tuned model from Experiment 10.

In [ ]:
pred = load_predictions("exp10", "FinalModel")

print(f"Loaded {len(pred):,} test predictions")
print(f"Overall accuracy: {pred['correct'].mean():.4f}\n")
print("Subgroup sizes:")
for sg, n in pred["subgroup"].value_counts().items():
    print(f"  {sg:<14}: {n:>6,}")

print(f"\nReference (privileged) group: {REFERENCE_SUBGROUP}")
print("Favourable outcome: model predicted correctly")

Loaded 12,284 test predictions
Overall accuracy: 0.5981

Subgroup sizes:
  formal        : 10,398
  other         :  1,116
  emoji-heavy   :    696
  slang-heavy   :     60
  sarcasm       :     14

Reference (privileged) group: formal
Favourable outcome: model predicted correctly


## Cell 3: AIF360 Audit

Four binary comparisons, each pitting one informal subgroup against the formal
reference.

In [ ]:
aif_rows = []
for sg in ["emoji-heavy", "slang-heavy", "sarcasm", "mixed"]:
    if sg not in pred["subgroup"].values:
        print(f"  {sg}: not present, skipping")
        continue
    r = {}
    r.update(fairness_aif360(pred, sg))
    r.update(fairness_classification_aif360(pred, sg))
    if r:
        aif_rows.append(r)
        print(f"{sg}:")
        print(f"  SPD {r.get('AIF360 SPD', float('nan')):+.4f}   "
              f"DIR {r.get('AIF360 DIR', float('nan')):.4f}   "
              f"EOD {r.get('AIF360 EOD', float('nan')):+.4f}   "
              f"AOD {r.get('AIF360 AOD', float('nan')):+.4f}")
        print(f"  Theil {r.get('Theil Index', float('nan')):.4f}   "
              f"Consistency {r.get('Consistency Score', float('nan')):.4f}\n")

aif_df = pd.DataFrame(aif_rows)

pip install 'aif360[inFairness]'


emoji-heavy:
  SPD +0.0287   DIR 1.0484   EOD -0.0040   AOD -0.0128
  Theil 0.5212   Consistency nan

slang-heavy:
  SPD +0.0246   DIR 1.0416   EOD +0.0405   AOD +0.0052
  Theil 0.5239   Consistency nan

sarcasm:
  SPD +0.1937   DIR 1.3271   EOD +0.2413   AOD +0.0666
  Theil 0.5237   Consistency nan

  mixed: not present, skipping


## Cell 4: Fairlearn Cross-Check

The same comparisons through Microsoft's implementation. Also produces the
MetricFrame table, which computes any sklearn metric per subgroup in one call.

In [ ]:
from fairlearn.metrics import MetricFrame
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

fl_rows = []
for sg in ["emoji-heavy", "slang-heavy", "sarcasm", "mixed"]:
    if sg not in pred["subgroup"].values:
        continue
    r = fairness_fairlearn(pred, sg)
    if r:
        fl_rows.append(r)
        print(f"{sg}: DPD {r.get('Fairlearn DPD', float('nan')):+.4f}   "
              f"EOD {r.get('Fairlearn EOD', float('nan')):+.4f}")

fl_df = pd.DataFrame(fl_rows)

print("\n" + "="*70)
print("FAIRLEARN METRICFRAME — ALL SUBGROUPS IN ONE CALL")
print("="*70)

mf = MetricFrame(
    metrics={
        "accuracy":  accuracy_score,
        "precision": lambda a, b: precision_score(a, b, average="macro", zero_division=0),
        "recall":    lambda a, b: recall_score(a, b, average="macro", zero_division=0),
        "f1_macro":  lambda a, b: f1_score(a, b, average="macro", zero_division=0),
    },
    y_true=pred["y_true"], y_pred=pred["y_pred"],
    sensitive_features=pred["subgroup"])

print(mf.by_group.round(4).to_string())
print(f"\nOverall: {mf.overall.round(4).to_dict()}")
print(f"\nMax minus min across subgroups:")
print(mf.difference().round(4).to_string())

save_result_table(mf.by_group.reset_index(), "Table9b_MetricFrame")

emoji-heavy: DPD +0.0287   EOD +0.0649
slang-heavy: DPD +0.0246   EOD +0.1530
sarcasm: DPD +0.1937   EOD +0.2936

FAIRLEARN METRICFRAME — ALL SUBGROUPS IN ONE CALL
             accuracy  precision  recall  f1_macro
subgroup                                          
emoji-heavy    0.6207     0.6281  0.5928    0.6022
formal         0.5920     0.5810  0.5968    0.5801
other          0.6371     0.5963  0.5852    0.5844
sarcasm        0.7857     0.7667  0.8381    0.7897
slang-heavy    0.6167     0.6298  0.6373    0.6180

Overall: {'accuracy': 0.5981, 'precision': 0.5915, 'recall': 0.5989, 'f1_macro': 0.5878}

Max minus min across subgroups:
accuracy     0.1937
precision    0.1857
recall       0.2528
f1_macro     0.2096
  saved table -> Table9b_MetricFrame.csv


PosixPath('/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/results/Table9b_MetricFrame.csv')

## Cell 5: Cross-Validation — the Point of the Exercise

Two toolkits, same question, independent code. Agreement means the finding
survives implementation choices.

In [ ]:
audit = full_fairness_audit(pred)

print("="*90)
print("FRAMEWORK AGREEMENT CHECK")
print("="*90)

for _, r in audit.iterrows():
    spd = r.get("AIF360 SPD", np.nan)
    dpd = r.get("Fairlearn DPD", np.nan)
    print(f"\n{r['Subgroup']}:")
    print(f"  AIF360 SPD    : {spd:+.4f}")
    print(f"  Fairlearn DPD : {dpd:+.4f}")
    if not (np.isnan(spd) or np.isnan(dpd)):
        diff = abs(abs(spd) - abs(dpd))
        print(f"  difference    : {diff:.4f}")
        if diff < 0.02:
            print("  -> AGREE. Finding is robust across implementations.")
        else:
            print("  -> DISAGREE. Report as an implementation observation,")
            print("     do not silently pick one.")

agree = (audit["Frameworks Agree?"] == "Yes").sum()
total = len(audit[audit["Frameworks Agree?"] != "N/A"])
print(f"\nAgreement: {agree} of {total} comparisons")

FRAMEWORK AGREEMENT CHECK

emoji-heavy:
  AIF360 SPD    : +0.0287
  Fairlearn DPD : +0.0287
  difference    : 0.0000
  -> AGREE. Finding is robust across implementations.

slang-heavy:
  AIF360 SPD    : +0.0246
  Fairlearn DPD : +0.0246
  difference    : 0.0000
  -> AGREE. Finding is robust across implementations.

sarcasm:
  AIF360 SPD    : +0.1937
  Fairlearn DPD : +0.1937
  difference    : 0.0000
  -> AGREE. Finding is robust across implementations.

Agreement: 3 of 3 comparisons


## Cell 6: Table 9

In [ ]:
table9 = audit.copy()
for c_ in ["Theil Index", "Consistency Score"]:
    if c_ not in table9.columns:
        table9[c_] = np.nan

cols = ["Subgroup", "N", "AIF360 SPD", "AIF360 DIR", "AIF360 EOD", "AIF360 AOD",
        "Theil Index", "Consistency Score", "Fairlearn DPD", "Fairlearn EOD",
        "Frameworks Agree?", "Passes Threshold?"]
table9 = table9[[c_ for c_ in cols if c_ in table9.columns]]

print("="*110)
print("TABLE 9 — DUAL FRAMEWORK BIAS AUDIT")
print("="*110)
print(table9.round(4).to_string(index=False))

print("\nThresholds: |SPD| < 0.10 | DIR 0.80-1.25 | |EOD| < 0.10 |")
print("            |AOD| < 0.10 | Consistency > 0.80")

failing = table9[table9["Passes Threshold?"].astype(str).str.startswith("No")]
if len(failing):
    print(f"\n{len(failing)} subgroup(s) fail at least one threshold:")
    for _, r in failing.iterrows():
        print(f"  {r['Subgroup']}: {r['Passes Threshold?']}")
else:
    print("\nAll subgroups pass all thresholds.")

save_result_table(table9, "Table9_Dual_Framework_Audit")
print("\nNovelty 6 complete.")

TABLE 9 — DUAL FRAMEWORK BIAS AUDIT
   Subgroup   N  AIF360 SPD  AIF360 DIR  AIF360 EOD  AIF360 AOD  Theil Index  Consistency Score  Fairlearn DPD  Fairlearn EOD Frameworks Agree? Passes Threshold?
emoji-heavy 696      0.0287      1.0484     -0.0040     -0.0128       0.5212                NaN         0.0287         0.0649               Yes               Yes
slang-heavy  60      0.0246      1.0416      0.0405      0.0052       0.5239                NaN         0.0246         0.1530               Yes               Yes
    sarcasm  14      0.1937      1.3271      0.2413      0.0666       0.5237                NaN         0.1937         0.2936               Yes          No - DIR

Thresholds: |SPD| < 0.10 | DIR 0.80-1.25 | |EOD| < 0.10 |
            |AOD| < 0.10 | Consistency > 0.80

1 subgroup(s) fail at least one threshold:
  sarcasm: No - DIR
  saved table -> Table9_Dual_Framework_Audit.csv

Novelty 6 complete.
